# Day 29 - Structured Streaming Lakehouse Pattern

**Topic:** Structured Streaming Lakehouse Pattern  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึก pattern การ ingest streaming events เข้า Bronze Lakehouse table ด้วย checkpoint, audit columns, restart mindset และ monitoring เบื้องต้น

วันนี้เราจะต่อจาก Structured Streaming basics ไปสู่ Lakehouse pattern ที่เจอบ่อยในงาน Data Engineering จริง คือรับ events แบบ incremental/micro-batch แล้วเขียนเข้า Bronze table พร้อม checkpoint เพื่อให้ restart ได้ปลอดภัยขึ้น จากนั้นค่อยอ่าน Bronze ไปทำ Silver แบบ batch-style เพื่อควบคุม data quality และ dedup ได้ง่ายขึ้น

## Goal of the day

1. เข้าใจว่า **Structured Streaming Lakehouse pattern** ใช้ต่างจาก batch Bronze ingestion อย่างไร
2. อธิบายได้ว่า **checkpoint location** สำคัญต่อ progress tracking และ recovery อย่างไร
3. สร้าง streaming ingestion เข้า Bronze Delta table ได้ด้วย PySpark
4. เพิ่ม audit columns เช่น `ingestion_timestamp`, `source_system`, `input_file_name`, `micro_batch_id` ได้
5. เข้าใจ trade-off เรื่อง trigger interval, small files, schema handling และ monitoring

## Why it matters in production

ใน production pipeline streaming ไม่ใช่แค่การเขียน code ให้รับข้อมูลใหม่ได้เรื่อย ๆ แต่ต้องออกแบบให้ recover, monitor และ reprocess ได้อย่างมีวินัย เพราะ:

- streaming job อาจถูก restart ได้ตลอดเวลา ไม่ว่าจะจาก failure, cluster restart หรือ deployment
- checkpoint ช่วยเก็บ progress/state เพื่อให้ Spark รู้ว่าอ่าน source ไปถึงไหนแล้ว
- Bronze table เป็นจุดรับ raw events ที่ควรเก็บ traceability ไว้ให้ตรวจย้อนหลังได้
- trigger ที่ถี่เกินไปอาจสร้าง small files จำนวนมากใน Lakehouse
- schema ของ events อาจเปลี่ยน และถ้าไม่คุมให้ดี streaming query อาจ fail หรือเขียนข้อมูลที่มี schema ไม่ตรงเข้า Bronze
- downstream Silver/Gold อาจต้อง deduplicate เพราะ streaming sink แบบ append อาจยังเจอ duplicate จาก source หรือ rerun strategy ได้

Production takeaway:

> Streaming Lakehouse ที่ไว้ใจได้ต้องออกแบบทั้ง source, sink, checkpoint, schema, dedup, monitoring และ cleanup ไม่ใช่ดูแค่ `.readStream` / `.writeStream` syntax

## Key concepts

| Concept | Meaning |
|---|---|
| Streaming ingestion to Bronze | รับ events ใหม่แบบ incremental/micro-batch แล้ว append เข้า Bronze table เพื่อเก็บ raw event records |
| Checkpoint location | path ที่ Spark ใช้เก็บ progress/state ของ streaming query เช่น files ที่อ่านแล้ว, offsets, metadata |
| Append mode | output mode ที่เพิ่ม rows ใหม่เข้า sink เหมาะกับ event ingestion ที่ไม่ต้อง update rows เดิม |
| Trigger | การกำหนดว่า Spark จะ process micro-batch เมื่อไหร่และบ่อยแค่ไหน เช่น run once สำหรับ lab หรือ interval สำหรับ production |
| Micro-batch | Spark แบ่งข้อมูลใหม่ที่เข้ามาเป็น batch เล็ก ๆ แล้ว execute query incrementally |
| `foreachBatch` | pattern ที่ให้เรา process แต่ละ micro-batch ด้วย batch DataFrame logic เช่น add `micro_batch_id` หรือ write log |
| Audit columns | technical columns ที่ช่วย trace เช่น `ingestion_timestamp`, `source_system`, `input_file_name`, `micro_batch_id` |
| Small files risk | trigger ที่ถี่เกินไปหรือ data volume ต่อ micro-batch น้อย ทำให้เกิดไฟล์เล็กจำนวนมากใน Bronze table ส่งผลต่อ read performance และ metadata overhead |
| Restart behavior | หลัง restart Spark ใช้ checkpoint เพื่อ resume จากจุดที่หยุดไว้ ไม่ต้องอ่าน source ซ้ำตั้งแต่ต้น ถ้า checkpoint และ sink ออกแบบมาถูกต้อง |


## Concept explanation

### Structured Streaming Lakehouse pattern คืออะไร?

ใน batch Bronze ingestion เรามักอ่านไฟล์หรือ source เป็นรอบ ๆ แล้วเขียนเข้า Bronze table ด้วย `.write` เช่น:

```python
df.write.mode("append").format("delta").saveAsTable("bronze_events")
```

แต่ใน streaming ingestion เราจะสร้าง streaming DataFrame จาก `.readStream` แล้วเริ่มรัน streaming query ด้วย `.writeStream.start()` เพื่อให้ Spark process ข้อมูลใหม่แบบ incremental:

```python
stream_df = spark.readStream.schema(event_schema).json(input_path)
query = stream_df.writeStream.start()
```

หลักคิดคือ:

> Batch Bronze = โหลดข้อมูลเป็นรอบ  
> Streaming Bronze = รับข้อมูลใหม่เรื่อย ๆ ผ่าน micro-batches

### ทำไม Bronze ควรเป็น append-first?

สำหรับ event data ส่วนใหญ่ Bronze ควรเป็น append-first เพราะ Bronze มีหน้าที่เก็บหลักฐานว่า source ส่งอะไรมา ไม่ควรรีบตีความหรือ overwrite ความจริงจาก source เร็วเกินไป

ใน lab นี้ Bronze table จะมี:

- event fields จาก source
- `event_timestamp` ที่ parse จาก string
- `ingestion_timestamp` ตอน Spark รับเข้า Lakehouse
- `source_system` เพื่อบอกว่า event มาจากระบบไหน
- `input_file_name` เพื่อ trace กลับไปที่ไฟล์
- `micro_batch_id` เพื่อรู้ว่า record ถูก process ใน micro-batch ไหน

### Checkpoint สำคัญตรงไหน?

Checkpoint คือ memory ระยะยาวของ streaming query

ถ้าไม่มี checkpoint หรือ checkpoint ถูกลบ Spark อาจไม่รู้ว่า:

- อ่านไฟล์ไหนไปแล้ว
- offset ไหนถูก process แล้ว
- state ของ aggregation/window อยู่ตรงไหน
- restart แล้วควรเริ่มจากตำแหน่งใด

ใน lab นี้เราจะใช้ checkpoint path แยกจาก data path เพื่อให้เห็นภาพว่า:

> Data path = output ที่ query หรือ downstream process อ่าน     
> Checkpoint path = operational state ที่ Spark ใช้ track progress และ recover

### ทำไมใช้ `trigger(once=True)` ใน lab?

Production streaming job อาจ run ต่อเนื่องด้วย trigger interval เช่น ทุก 1 นาที หรือทุก 5 นาที แต่ใน notebook learning lab ถ้าใช้ continuous query ยาว ๆ อาจทำให้ run all ค้าง

ดังนั้นวันนี้ใช้ `trigger(once=True)` เพื่อให้ Spark process files ที่มีอยู่ตอนนั้นทั้งหมดใน micro-batch เดียว แล้วหยุด query เองโดยอัตโนมัติ

ข้อดี:

- เหมาะกับ notebook lab
- เห็น checkpoint behavior ได้
- ไม่ต้องปล่อย query running ค้างไว้

ข้อควรจำ:

> `trigger(once=True)` ยังเป็น Structured Streaming query แต่รันแบบ bounded micro-batch เพื่อให้ lab จบเอง

### Bronze streaming แล้ว Silver ทำอย่างไร?

Pattern ที่เข้าใจง่ายสำหรับ learning lab คือ:

1. Streaming เขียนเข้า Bronze แบบ append
2. อ่าน Bronze เป็น batch DataFrame
3. Deduplicate / validate / standardize เป็น Silver
4. แยก invalid records ถ้าจำเป็น

วิธีนี้ช่วยให้เราแยก concern ได้ชัด:

- Streaming layer เน้น reliable ingestion
- Silver layer เน้น business/data quality logic


## Mermaid diagram: Streaming Lakehouse Bronze Pattern

```mermaid
flowchart LR
    A[Mock Event Files] --> B[readStream with explicit schema]
    B --> C[Add audit columns]
    C --> D[writeStream foreachBatch]
    D --> E[Bronze Delta Table]
    D --> F[Micro-batch Log Table]
    G[Checkpoint Path] <--> D
    E --> H[Batch Silver Cleansing]
    H --> I[Silver Valid Table]
    H --> J[Silver Invalid Table]
    K[Monitoring] --> D
    K --> F
```

Key idea:

> Streaming job ควรมีทั้ง output table, checkpoint path และ monitoring/logging ไม่ใช่มีแค่ sink อย่างเดียว

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

from datetime import datetime
import json
import os
import shutil  # Built-in file and directory utilities

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 3, Finished, Available, Finished, False)

## Create mock streaming source data

ใน lab นี้จะใช้ file source เป็น mock streaming source เพราะรันใน Fabric Notebook ได้ง่ายและควบคุม behavior ได้ชัด

แนวคิดคือ:

1. เขียน JSON events batch แรกลง folder `input`
2. ใช้ `readStream` อ่านจาก folder นี้
3. เขียนเข้า Bronze table พร้อม checkpoint
4. เพิ่ม JSON file batch ใหม่
5. restart streaming query ด้วย checkpoint เดิม แล้วตรวจว่า Spark อ่านเฉพาะ file ใหม่

> หมายเหตุ: cell reset ด้านล่างใช้เฉพาะ path/table ของ Day 29 เพื่อให้ notebook rerun ได้ง่าย ใน production ห้ามลบ checkpoint เพื่อ reset job โดยไม่วางแผน เพราะอาจเกิด reprocess หรือ duplicate ได้

In [2]:
# Fabric has two useful path styles:
# - Mount path: use with Python file APIs such as open(), os, and shutil.
# - Spark path: use with spark.read / spark.write.
base_path_mount = "/lakehouse/default/Files/day29_structured_streaming_lakehouse_pattern"
base_path_spark = "Files/day29_structured_streaming_lakehouse_pattern"

input_path_mount = f"{base_path_mount}/input"
input_path_spark = f"{base_path_spark}/input"
checkpoint_path_spark = f"{base_path_spark}/checkpoint/bronze_events"

bronze_table = "day29_bronze_streaming_events"
silver_table = "day29_silver_streaming_events"
invalid_table = "day29_invalid_streaming_events"
micro_batch_log_table = "day29_streaming_micro_batch_log"

# Lab-only reset for repeatable notebook runs.
# Do not delete production checkpoints unless you intentionally want to reset processing progress.
for table_name in [bronze_table, silver_table, invalid_table, micro_batch_log_table]:
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")

if os.path.exists(base_path_mount):
    shutil.rmtree(base_path_mount)

os.makedirs(input_path_mount, exist_ok=True)

print("Input path (mount):", input_path_mount)
print("Checkpoint path (spark):", checkpoint_path_spark)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 4, Finished, Available, Finished, False)

Input path (mount): /lakehouse/default/Files/day29_structured_streaming_lakehouse_pattern/input
Checkpoint path (spark): Files/day29_structured_streaming_lakehouse_pattern/checkpoint/bronze_events


In [3]:
event_schema = T.StructType([
    T.StructField("event_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("event_time", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("source_system", T.StringType(), True),
])


def write_json_lines(records, file_path):
    """Write records as JSON lines for a streaming file source lab."""
    with open(file_path, "w", encoding="utf-8") as file:  # Write text using UTF-8 encoding
        for record in records:
            file.write(json.dumps(record) + "\n")  # Keep one JSON object per line for JSON Lines / NDJSON format


batch_01_records = [
    {
        "event_id": "E001",
        "customer_id": 101,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:00:00",
        "amount": 1200.50,
        "source_system": "mobile_app",
    },
    {
        "event_id": "E002",
        "customer_id": 102,
        "event_type": "refund",
        "event_time": "2026-06-14 09:01:00",
        "amount": -50.00,
        "source_system": "mobile_app",
    },
    {
        "event_id": "E003",
        "customer_id": 103,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:02:00",
        "amount": 500.00,
        "source_system": "web_portal",
    },
    {
        "event_id": "E004",
        "customer_id": 104,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:03:00",
        "amount": 800.00,
        "source_system": "web_portal",
    },
]

write_json_lines(batch_01_records, f"{input_path_mount}/events_batch_01.json")

print("Created initial mock event file:")
print(os.listdir(input_path_mount))

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 5, Finished, Available, Finished, False)

Created initial mock event file:
['events_batch_01.json']


## PySpark code examples

ใน section นี้จะสร้าง flow แบบ streaming-to-Bronze แล้วต่อด้วย Bronze-to-Silver เพื่อให้เห็น Lakehouse pattern ครบ flow แต่ยังเล็กพอสำหรับ notebook lab

### Example 1: Create a streaming DataFrame from file source

`readStream` จะสร้าง streaming DataFrame พร้อม execution plan ไว้ก่อน ยังไม่ได้เริ่ม process จนกว่าจะเรียก `.writeStream.start()`

สิ่งที่ควรสังเกต:

- ต้องกำหนด schema ให้ชัดเจน
- `df_events_stream.isStreaming` ควรเป็น `True`
- การเรียก `printSchema()` ช่วยดู structure แต่ยังไม่ใช่การ process data

In [4]:
df_events_stream = (
    spark.readStream
    .schema(event_schema)
    .option("maxFilesPerTrigger", 1)
    .json(input_path_spark)
)

print("Is streaming DataFrame:", df_events_stream.isStreaming)
df_events_stream.printSchema()

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 6, Finished, Available, Finished, False)

Is streaming DataFrame: True
root
 |-- event_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- source_system: string (nullable = true)



### Example 2: Add Bronze audit columns

ก่อนเขียนเข้า Bronze table เราจะเพิ่ม audit columns เพื่อให้ trace ได้ว่า record มาจากไหน และถูก ingest เมื่อไร

ใน streaming query เราสามารถเพิ่ม column ปกติได้เหมือน batch DataFrame แต่ต้องจำไว้ว่า column เช่น `current_timestamp()` จะถูก evaluate ตอน micro-batch execute จริง ไม่ใช่ตอนที่เขียน code สร้าง DataFrame

In [5]:
df_bronze_stream_input = (
    df_events_stream
    .withColumn("event_timestamp", F.to_timestamp("event_time"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("bronze_source", F.lit("mock_file_stream"))
    .withColumn("input_file_name", F.input_file_name())
)

print("Bronze streaming DataFrame with audit columns is ready.")
print("Still not started until writeStream.start() is called.")

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 7, Finished, Available, Finished, False)

Bronze streaming DataFrame with audit columns is ready.
Still not started until writeStream.start() is called.


### Example 3: Write streaming micro-batches into a Bronze table

ในตัวอย่างนี้ใช้ `foreachBatch` ซึ่งเป็น sink option ที่ให้ Spark เรียก custom function ของเราทุกครั้งที่ micro-batch พร้อม แทนที่จะเขียนตรงเข้า sink โดยตรง ทำให้ควบคุม write logic ได้ละเอียดขึ้น

เหตุผลที่ใช้ `foreachBatch`:

- เพิ่ม `micro_batch_id` ได้ง่าย
- เขียน Bronze table ด้วย `.write.format("delta").saveAsTable()` ได้
- เขียน micro-batch log ได้ใน cell เดียวกัน
- เข้าใจ micro-batch mindset ว่า Spark ส่ง batch DataFrame เล็กๆ เข้ามาให้ function ทีละ batch

> ใน production ต้องระวังว่า logic ใน `foreachBatch` ควร idempotent หรือมี dedup strategy เพราะถ้า retry แล้วเขียนซ้ำได้ อาจเกิด duplicate ใน sink

In [6]:
# foreachBatch calls this function once per micro-batch.
# The function must accept 2 parameters in this order:
# 1) micro_batch_df: DataFrame for the current micro-batch
# 2) micro_batch_id: unique id of the current micro-batch
def write_bronze_micro_batch(micro_batch_df, micro_batch_id):
    """Write one streaming micro-batch to Bronze and log basic metrics."""
    cached_batch_df = micro_batch_df.cache()  # Cache to avoid recomputing the micro-batch multiple times.
    row_count = cached_batch_df.count()  # Materialize cache and count rows in this micro-batch.

    if row_count > 0:
        df_to_write = (
            cached_batch_df
            .withColumn("micro_batch_id", F.lit(int(micro_batch_id)))
            .withColumn("micro_batch_processed_at", F.current_timestamp())
        )

        (
            df_to_write
            .write
            .mode("append")
            .format("delta")
            .saveAsTable(bronze_table)
        )

    log_rows = [(int(micro_batch_id), int(row_count), datetime.utcnow().isoformat())]
    log_schema = T.StructType([
        T.StructField("micro_batch_id", T.IntegerType(), False),
        T.StructField("input_row_count", T.IntegerType(), False),
        T.StructField("logged_at_utc", T.StringType(), False),
    ])

    df_log = spark.createDataFrame(log_rows, log_schema)

    (
        df_log
        .write
        .mode("append")
        .format("delta")
        .saveAsTable(micro_batch_log_table)
    )

    cached_batch_df.unpersist()

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 8, Finished, Available, Finished, False)

In [7]:
query_01 = (
    df_bronze_stream_input
    .writeStream
    .foreachBatch(write_bronze_micro_batch)
    .option("checkpointLocation", checkpoint_path_spark)
    .outputMode("append")
    .trigger(once=True)  # Process available data once, then stop automatically
    .start()
)

query_01.awaitTermination()  # Wait here until the streaming query finishes

print("Query 01 finished.")
print("Query id:", query_01.id)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 9, Finished, Available, Finished, False)

Query 01 finished.
Query id: 8c274148-60de-4ee5-a270-aee94a2e008f


### Example 4: Validate Bronze table and micro-batch log

หลัง query จบแล้ว เราควรตรวจอย่างน้อย 2 อย่าง:

1. Bronze table มี records ที่คาดไว้
2. Micro-batch log บอกได้ว่า batch ไหน process กี่ rows

ใน production อาจใช้ monitoring tool หรือ pipeline log แทนการ `.show()` ใน notebook

In [8]:
df_bronze_after_batch_01 = spark.read.table(bronze_table)

print("Bronze row count after batch 01:", df_bronze_after_batch_01.count())

df_bronze_after_batch_01.orderBy("event_id", "micro_batch_id").show(truncate=False)
spark.read.table(micro_batch_log_table).orderBy("micro_batch_id").show(truncate=False)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 10, Finished, Available, Finished, False)

Bronze row count after batch 01: 4
+--------+-----------+----------+-------------------+------+-------------+-------------------+-----------------------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+--------------------------+
|event_id|customer_id|event_type|event_time         |amount|source_system|event_timestamp    |ingestion_timestamp    |bronze_source   |input_file_name                                                                                                                                                                                 |micro_batch_id|micro_batch_processed_at  |
+--------+-----------+----------+-------------------+------+-------------+-------------------+-----------------------+----------------+----------------------------------------------------------------------------------------

### Example 5: Simulate restart with the same checkpoint

ตอนนี้เราจะเพิ่มไฟล์ batch ใหม่ แล้วสร้าง streaming query ใหม่โดยใช้ checkpoint path เดิม

Expected behavior:

- Spark ควร process เฉพาะไฟล์ใหม่
- ไฟล์ batch 01 ไม่ควรถูก process ซ้ำ เพราะ checkpoint จำ progress ไว้แล้ว
- Bronze table จะมี rows เพิ่มจาก batch 02

> ถ้าลบ checkpoint แล้ว run ใหม่ Spark อาจอ่านไฟล์เก่าซ้ำได้ ดังนั้นใน production ต้องจัดการ checkpoint อย่างระวังมาก

In [9]:
batch_02_records = [
    {
        "event_id": "E005",
        "customer_id": 105,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:04:00",
        "amount": 300.00,
        "source_system": "mobile_app",
    },
    {
        "event_id": "E006",
        "customer_id": 106,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:05:00",
        "amount": 0.00,
        "source_system": "web_portal",
    },
    {
        "event_id": "E004",
        "customer_id": 104,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:06:00",
        "amount": 850.00,
        "source_system": "web_portal",
    },
    {
        "event_id": "E007",
        "customer_id": 107,
        "event_type": "purchase",
        "event_time": "2026-06-14 08:50:00",
        "amount": 999.00,
        "source_system": "partner_api",
    },
]

write_json_lines(batch_02_records, f"{input_path_mount}/events_batch_02.json")

print("Current input files:")
print(sorted(os.listdir(input_path_mount)))  # sorted() keeps the file list order consistent

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 11, Finished, Available, Finished, False)

Current input files:
['events_batch_01.json', 'events_batch_02.json']


In [10]:
df_events_stream_restart = (
    spark.readStream
    .schema(event_schema)
    .option("maxFilesPerTrigger", 1)
    .json(input_path_spark)
)

df_bronze_stream_restart_input = (
    df_events_stream_restart
    .withColumn("event_timestamp", F.to_timestamp("event_time"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("bronze_source", F.lit("mock_file_stream"))
    .withColumn("input_file_name", F.input_file_name())
)

query_02 = (
    df_bronze_stream_restart_input
    .writeStream
    .foreachBatch(write_bronze_micro_batch)
    .option("checkpointLocation", checkpoint_path_spark)
    .outputMode("append")
    .trigger(once=True)
    .start()
)

query_02.awaitTermination()

print("Query 02 finished.")
print("Query id:", query_02.id)  # Same query id as query_01 when using the same checkpoint; new checkpoint means new query id

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 12, Finished, Available, Finished, False)

Query 02 finished.
Query id: 8c274148-60de-4ee5-a270-aee94a2e008f


In [11]:
df_bronze_after_batch_02 = spark.read.table(bronze_table)

print("Bronze row count after batch 02:", df_bronze_after_batch_02.count())

df_bronze_after_batch_02.orderBy("event_id", "micro_batch_id", "ingestion_timestamp").show(truncate=False)
spark.read.table(micro_batch_log_table).orderBy("micro_batch_id").show(truncate=False)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 13, Finished, Available, Finished, False)

Bronze row count after batch 02: 8
+--------+-----------+----------+-------------------+------+-------------+-------------------+-----------------------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+--------------------------+
|event_id|customer_id|event_type|event_time         |amount|source_system|event_timestamp    |ingestion_timestamp    |bronze_source   |input_file_name                                                                                                                                                                                 |micro_batch_id|micro_batch_processed_at  |
+--------+-----------+----------+-------------------+------+-------------+-------------------+-----------------------+----------------+----------------------------------------------------------------------------------------

### Example 6: Build Silver from streaming Bronze

ใน lab นี้ Silver จะถูกสร้างแบบ batch โดยอ่านจาก Bronze เพื่อให้เห็น pattern การแยก layer ได้ชัดเจน:

- deduplicate ด้วย `event_id`
- เลือก record ล่าสุดตาม `ingestion_timestamp` และ `micro_batch_id`
- validate field สำคัญ เช่น `amount > 0` และ `event_timestamp is not null`
- แยก valid table และ invalid/review table

Pattern นี้เหมาะสำหรับเริ่มต้น เพราะ streaming query รับผิดชอบเฉพาะการ ingest event เข้า Bronze ก่อน ส่วน business cleansing, deduplication และ validation ค่อยทำใน Silver แบบ batch ทำให้ logic อ่านง่ายกว่า debug ง่ายกว่า และลดความซับซ้อนของ streaming job

In [12]:
window_latest_event = (
    Window
    .partitionBy("event_id")
    .orderBy(F.col("ingestion_timestamp").desc(), F.col("micro_batch_id").desc())
)

df_bronze_events = spark.read.table(bronze_table)

df_latest_events = (
    df_bronze_events
    .withColumn("row_number_per_event", F.row_number().over(window_latest_event))
    .filter(F.col("row_number_per_event") == 1)
    .drop("row_number_per_event")
)

df_classified_events = (
    df_latest_events
    .withColumn(
        "dq_issue_array_raw",
        F.array(
            # F.when() returns null when the condition is not met.
            # So this raw array always has 4 elements, e.g. ["missing_event_id", null, null, null].
            F.when(F.col("event_id").isNull(), F.lit("missing_event_id")),
            F.when(F.col("customer_id").isNull(), F.lit("missing_customer_id")),
            F.when(F.col("event_timestamp").isNull(), F.lit("invalid_event_timestamp")),
            F.when(F.col("amount").isNull() | (F.col("amount") <= 0), F.lit("invalid_amount")),
        ),
    )
    .withColumn("dq_issue_array", F.expr("filter(dq_issue_array_raw, x -> x is not null)"))  # Filter out nulls
    .drop("dq_issue_array_raw")
    .withColumn(
        "record_status",
        F.when(F.size(F.col("dq_issue_array")) == 0, F.lit("valid")).otherwise(F.lit("invalid")),
    )
)

# Tips:
# - array(value1, value2, ...) combines multiple values into one array.
# - expr("SQL expression") lets you use Spark SQL functions inside PySpark DataFrame code.
# - Higher-order function: a function that applies logic to elements inside an array.
#   Example: filter(array_column, element -> condition) keeps only array elements that match the condition.
# - Lambda expression / anonymous function: inline logic defined without a separate function name.
#   Example: element -> condition defines the filtering logic for each array element.

df_silver_valid_events = (
    df_classified_events
    .filter(F.col("record_status") == "valid")
    .select(
        "event_id",
        "customer_id",
        "event_type",
        "event_timestamp",
        "amount",
        "source_system",
        "ingestion_timestamp",
        "micro_batch_id",
    )
)

df_invalid_events = (
    df_classified_events
    .filter(F.col("record_status") == "invalid")
    .select(
        "event_id",
        "customer_id",
        "event_type",
        "event_time",
        "event_timestamp",
        "amount",
        "source_system",
        "dq_issue_array",
        "ingestion_timestamp",
        "micro_batch_id",
        "input_file_name",
    )
)

(
    df_silver_valid_events
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

(
    df_invalid_events
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(invalid_table)
)

print("Silver valid rows:", spark.read.table(silver_table).count())
print("Invalid/review rows:", spark.read.table(invalid_table).count())

spark.read.table(silver_table).orderBy("event_id").show(truncate=False)
spark.read.table(invalid_table).orderBy("event_id").show(truncate=False)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 14, Finished, Available, Finished, False)

Silver valid rows: 5
Invalid/review rows: 2
+--------+-----------+----------+-------------------+------+-------------+-----------------------+--------------+
|event_id|customer_id|event_type|event_timestamp    |amount|source_system|ingestion_timestamp    |micro_batch_id|
+--------+-----------+----------+-------------------+------+-------------+-----------------------+--------------+
|E001    |101        |purchase  |2026-06-14 09:00:00|1200.5|mobile_app   |2026-06-14 13:15:40.515|0             |
|E003    |103        |purchase  |2026-06-14 09:02:00|500.0 |web_portal   |2026-06-14 13:15:40.515|0             |
|E004    |104        |purchase  |2026-06-14 09:06:00|850.0 |web_portal   |2026-06-14 13:31:52.663|1             |
|E005    |105        |purchase  |2026-06-14 09:04:00|300.0 |mobile_app   |2026-06-14 13:31:52.663|1             |
|E007    |107        |purchase  |2026-06-14 08:50:00|999.0 |partner_api  |2026-06-14 13:31:52.663|1             |
+--------+-----------+----------+-----------

### Example 7: Inspect query progress and monitoring data

Structured Streaming มี metadata ของ query เช่น `lastProgress` ที่ช่วยดูว่า query ล่าสุด process อะไรไปบ้าง

ใน lab นี้เราเก็บ micro-batch log เพิ่มเองด้วย เพื่อให้เห็น pattern ง่าย ๆ ว่า streaming job ควรมี operational visibility

In [13]:
print("Last progress from query 02:")
print(query_02.lastProgress)

print("Micro-batch log table:")
spark.read.table(micro_batch_log_table).orderBy("micro_batch_id").show(truncate=False)

print("Bronze rows by micro_batch_id:")
(
    spark.read.table(bronze_table)
    .groupBy("micro_batch_id")
    .agg(F.count("*").alias("bronze_row_count"))
    .orderBy("micro_batch_id")
    .show()
)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 15, Finished, Available, Finished, False)

Last progress from query 02:
{'id': '8c274148-60de-4ee5-a270-aee94a2e008f', 'runId': '2a2ebd6e-e698-4e3b-a7b6-b9faf759cdf0', 'name': None, 'timestamp': '2026-06-14T13:31:52.007Z', 'batchId': 1, 'numInputRows': 4, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.5457025920873124, 'durationMs': {'addBatch': 5884, 'commitOffsets': 289, 'getBatch': 169, 'latestOffset': 391, 'queryPlanning': 38, 'triggerExecution': 7330, 'walCommit': 292}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Files/day29_structured_streaming_lakehouse_pattern/input]', 'startOffset': {'logOffset': 0}, 'endOffset': {'logOffset': 1}, 'latestOffset': None, 'numInputRows': 4, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.5457025920873124}], 'sink': {'description': 'ForeachBatchSink', 'numOutputRows': -1}}
Micro-batch log table:
+--------------+---------------+-----------

## Quick recap

| Question | Answer |
|---|---|
| Streaming Bronze ต่างจาก batch Bronze อย่างไร? | Streaming Bronze รับข้อมูลใหม่แบบ incremental/micro-batch ผ่าน `readStream` / `writeStream` |
| Checkpoint ใช้ทำอะไร? | เก็บ progress/state เพื่อให้ query restart แล้วรู้ว่าอ่าน source ถึงไหนแล้ว |
| ทำไม lab นี้ใช้ `trigger(once=True)`? | เพื่อให้ streaming query process ข้อมูลที่มีอยู่แล้วจบเอง เหมาะกับ notebook lab |
| ทำไมต้องมี audit columns? | เพื่อ trace source, ingestion time, input file และ micro-batch ได้ |
| ทำไมต้องระวัง small files? | trigger ที่ถี่เกินไปหรือ data volume ต่อ micro-batch น้อย อาจสร้างไฟล์เล็กจำนวนมาก ทำให้ read ช้าลงและ metadata overhead สูงขึ้น |
| Silver ควรทำอะไรต่อจาก Bronze? | Deduplicate, validate, standardize และแยก invalid/review records ตาม rule |

## Exercises

### Exercise 1: Add a new event file and process it with the same checkpoint

เพิ่ม JSON file batch ที่ 3 ลงใน `input_path_spark` แล้ว restart streaming query ด้วย `checkpoint_path_spark` เดิม

Requirements:

- สร้าง records อย่างน้อย 3 rows
- มีอย่างน้อย 1 row ที่ valid
- มีอย่างน้อย 1 row ที่ invalid เช่น `amount <= 0`
- ใช้ `write_bronze_micro_batch` function เดิม
- ตรวจ row count ของ Bronze หลัง run

Expected result:

- Bronze table มี records เพิ่มเฉพาะจากไฟล์ใหม่
- micro-batch log มี entry เพิ่ม
- ไฟล์ batch 01 และ batch 02 ไม่ควรถูก process ซ้ำถ้า checkpoint เดิมยังอยู่

In [14]:
batch_03_records = [
    {
        "event_id": "E008",
        "customer_id": 108,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:07:00",
        "amount": 1500.00,
        "source_system": "mobile_app",
    },
    {
        "event_id": "E009",
        "customer_id": 109,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:08:00",
        "amount": -20.00,
        "source_system": "partner_api",
    },
    {
        "event_id": "E010",
        "customer_id": 110,
        "event_type": "purchase",
        "event_time": "2026-06-14 09:09:00",
        "amount": 700.00,
        "source_system": "web_portal",
    },
]

write_json_lines(batch_03_records, f"{input_path_mount}/events_batch_03.json")

exercise_stream_df = (
    spark.readStream
    .schema(event_schema)
    .option("maxFilesPerTrigger", 1)
    .json(input_path_spark)
)

exercise_bronze_input_df = (
    exercise_stream_df
    .withColumn("event_timestamp", F.to_timestamp("event_time"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("bronze_source", F.lit("mock_file_stream"))
    .withColumn("input_file_name", F.input_file_name())
)

query_03 = (
    exercise_bronze_input_df
    .writeStream
    .foreachBatch(write_bronze_micro_batch)
    .option("checkpointLocation", checkpoint_path_spark)
    .outputMode("append")
    .trigger(once=True)
    .start()
)

query_03.awaitTermination()

print("Query 03 finished.")
print("Bronze row count after batch 03:", spark.read.table(bronze_table).count())

spark.read.table(micro_batch_log_table).orderBy("micro_batch_id").show(truncate=False)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 16, Finished, Available, Finished, False)

Query 03 finished.
Bronze row count after batch 03: 11
+--------------+---------------+--------------------------+
|micro_batch_id|input_row_count|logged_at_utc             |
+--------------+---------------+--------------------------+
|0             |4              |2026-06-14T13:15:57.956683|
|1             |4              |2026-06-14T13:31:57.077057|
|2             |3              |2026-06-14T14:01:41.232946|
+--------------+---------------+--------------------------+



### Exercise 2: Rebuild Silver after batch 03

ใช้ Bronze table ล่าสุด แล้ว rebuild Silver/Invalid tables อีกครั้ง

Requirements:

- deduplicate ด้วย `event_id`
- valid record ต้องมี `event_timestamp` และ `amount > 0`
- overwrite `silver_table` และ `invalid_table`
- แสดงจำนวน rows ของ valid และ invalid

Expected result:

- `E009` ควรถูกจัดเป็น invalid เพราะ `amount < 0`
- valid table ควรมีเฉพาะ events ล่าสุดต่อ `event_id`

In [15]:
df_bronze_latest = spark.read.table(bronze_table)

window_latest_event = (
    Window
    .partitionBy("event_id")
    .orderBy(F.col("ingestion_timestamp").desc(), F.col("micro_batch_id").desc())
)

df_latest_events_exercise = (
    df_bronze_latest
    .withColumn("row_number_per_event", F.row_number().over(window_latest_event))
    .filter(F.col("row_number_per_event") == 1)
    .drop("row_number_per_event")
)

df_classified_exercise = (
    df_latest_events_exercise
    .withColumn(
        "dq_issue_array_raw",
        F.array(
            F.when(F.col("event_id").isNull(), F.lit("missing_event_id")),
            F.when(F.col("customer_id").isNull(), F.lit("missing_customer_id")),
            F.when(F.col("event_timestamp").isNull(), F.lit("invalid_event_timestamp")),
            F.when(F.col("amount").isNull() | (F.col("amount") <= 0), F.lit("invalid_amount")),
        ),
    )
    .withColumn("dq_issue_array", F.expr("filter(dq_issue_array_raw, x -> x is not null)"))
    .drop("dq_issue_array_raw")
    .withColumn(
        "record_status",
        F.when(F.size(F.col("dq_issue_array")) == 0, F.lit("valid")).otherwise(F.lit("invalid")),
    )
)

(
    df_classified_exercise
    .filter(F.col("record_status") == "valid")
    .select(
        "event_id",
        "customer_id",
        "event_type",
        "event_timestamp",
        "amount",
        "source_system",
        "ingestion_timestamp",
        "micro_batch_id",
    )
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

(
    df_classified_exercise
    .filter(F.col("record_status") == "invalid")
    .select(
        "event_id",
        "customer_id",
        "event_type",
        "event_time",
        "event_timestamp",
        "amount",
        "source_system",
        "dq_issue_array",
        "ingestion_timestamp",
        "micro_batch_id",
        "input_file_name",
    )
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(invalid_table)
)

print("Silver valid rows after batch 03:", spark.read.table(silver_table).count())
print("Invalid rows after batch 03:", spark.read.table(invalid_table).count())

spark.read.table(silver_table).orderBy("event_id").show(truncate=False)
spark.read.table(invalid_table).orderBy("event_id").show(truncate=False)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 17, Finished, Available, Finished, False)

Silver valid rows after batch 03: 7
Invalid rows after batch 03: 3
+--------+-----------+----------+-------------------+------+-------------+-----------------------+--------------+
|event_id|customer_id|event_type|event_timestamp    |amount|source_system|ingestion_timestamp    |micro_batch_id|
+--------+-----------+----------+-------------------+------+-------------+-----------------------+--------------+
|E001    |101        |purchase  |2026-06-14 09:00:00|1200.5|mobile_app   |2026-06-14 13:15:40.515|0             |
|E003    |103        |purchase  |2026-06-14 09:02:00|500.0 |web_portal   |2026-06-14 13:15:40.515|0             |
|E004    |104        |purchase  |2026-06-14 09:06:00|850.0 |web_portal   |2026-06-14 13:31:52.663|1             |
|E005    |105        |purchase  |2026-06-14 09:04:00|300.0 |mobile_app   |2026-06-14 13:31:52.663|1             |
|E007    |107        |purchase  |2026-06-14 08:50:00|999.0 |partner_api  |2026-06-14 13:31:52.663|1             |
|E008    |108        

### Exercise 3: Create a simple streaming monitoring summary

สร้าง summary จาก micro-batch log เพื่อดูว่าแต่ละ micro-batch process กี่ rows

Requirements:

- อ่าน `micro_batch_log_table`
- คำนวณ total rows จากทุก micro-batch
- แสดง micro-batch ที่มี input rows มากกว่า 0
- แสดงจำนวน micro-batches ทั้งหมด

Expected result:

- เห็น micro-batch log ที่ช่วยตอบได้ว่า job process อะไรไปแล้วบ้าง
- เริ่มเห็นว่า streaming job ควรมี operational metrics ไม่ใช่ดูแค่ output table

In [16]:
df_micro_batch_log = spark.read.table(micro_batch_log_table)

print("Total micro-batches:", df_micro_batch_log.count())

(
    df_micro_batch_log
    .agg(F.sum("input_row_count").alias("total_input_rows"))
    .show()
)

(
    df_micro_batch_log
    .filter(F.col("input_row_count") > 0)
    .orderBy("micro_batch_id")
    .show(truncate=False)
)

StatementMeta(, 2780a6df-8590-4930-852d-da5f382d7c77, 18, Finished, Available, Finished, False)

Total micro-batches: 3
+----------------+
|total_input_rows|
+----------------+
|              11|
+----------------+

+--------------+---------------+--------------------------+
|micro_batch_id|input_row_count|logged_at_utc             |
+--------------+---------------+--------------------------+
|0             |4              |2026-06-14T13:15:57.956683|
|1             |4              |2026-06-14T13:31:57.077057|
|2             |3              |2026-06-14T14:01:41.232946|
+--------------+---------------+--------------------------+



## Common mistakes

1. **ลบ checkpoint เพื่อแก้ปัญหาแบบไม่คิดผลกระทบ**

   ใน lab เราอาจ reset checkpoint เพื่อเริ่มตัวอย่างใหม่ได้ แต่ใน production การลบ checkpoint อาจทำให้เกิด reprocess, duplicate หรือเสีย state ของ aggregation ได้

2. **ใช้ trigger ถี่เกินไปจนเกิด small files**

   ถ้า trigger ทุกไม่กี่วินาทีแต่ input น้อยมาก Lakehouse อาจมีไฟล์เล็กจำนวนมาก ต้องคิดเรื่อง compaction/OPTIMIZE และ trigger interval ให้เหมาะสม

3. **ไม่กำหนด schema ให้ streaming source**

   Streaming source ควรมี expected schema ที่ชัดเจน เพราะ schema inference กับ streaming source มักไม่เหมาะกับ production และทำให้ behavior ไม่แน่นอน

4. **เอา business logic หนัก ๆ ทั้งหมดไปใส่ใน streaming query ตั้งแต่แรก**

   Streaming step ควรเน้น reliable ingestion ก่อน ส่วน cleansing, dedup, DQ และ enrichment อาจแยกไปทำใน Silver pattern เพื่อให้ debug และ rerun ง่ายขึ้น

5. **คิดว่า checkpoint แปลว่าไม่มี duplicate แน่นอน 100%**

   Checkpoint ช่วยเรื่อง progress/recovery แต่ duplicate ยังเกิดได้จาก source retry, file resend, sink retry หรือ logic ใน `foreachBatch` ที่ไม่ idempotent จึงยังต้องมี dedup strategy ใน Silver หรือ downstream layer

6. **ไม่ monitor streaming job**

   Production streaming ต้องดู progress, input rows, processing time, lag, failures, checkpoint health และ output row count ไม่ใช่ดูแค่ว่า query ยัง running อยู่

7. **ใช้ output mode ไม่ตรงกับ logic**

   Append เหมาะกับ event ingestion ที่เพิ่ม rows ใหม่ ส่วน aggregation ที่ผลลัพธ์ยังเปลี่ยนได้ต้องใช้ update หรือ complete และบาง sink อาจไม่ support ทุก mode


## Expected Output / Success Criteria

เมื่อจบ Day 29 ควรทำได้:

- อธิบายได้ว่า Structured Streaming Lakehouse pattern คือการ ingest events เข้า Bronze table แบบ incremental/micro-batch
- ใช้ `readStream` กับ explicit schema เพื่อสร้าง streaming DataFrame ได้
- ใช้ `writeStream`, `foreachBatch`, `checkpointLocation`, `outputMode("append")` และ `trigger(once=True)` ใน lab ได้
- อธิบายได้ว่า checkpoint path ช่วยให้ streaming query restart แล้วจำ progress ได้อย่างไร
- เพิ่ม audit columns เช่น `event_timestamp`, `ingestion_timestamp`, `bronze_source`, `input_file_name`, `micro_batch_id` ได้
- เขียน streaming micro-batches เข้า Bronze Delta table ได้
- ตรวจ Bronze table และ micro-batch log ได้
- Simulate restart โดยเพิ่มไฟล์ใหม่และใช้ checkpoint เดิมได้
- สร้าง Silver table จาก streaming Bronze โดย deduplicate และ validate records ได้
- อธิบาย small files risk จาก trigger ที่ถี่เกินไปได้
- ตรวจสอบ streaming query progress และ micro-batch log เบื้องต้นได้
- แยกได้ว่า lab reset checkpoint กับ production checkpoint management เป็นคนละเรื่องกัน


## Final takeaway

Structured Streaming Lakehouse pattern ไม่ใช่แค่การเปลี่ยน `.read` เป็น `.readStream` แต่คือการออกแบบ ingestion flow ที่ recover และตรวจสอบได้

> Bronze streaming job ต้องมี checkpoint, audit columns, clear sink behavior, schema expectation, monitoring และ downstream dedup/recovery mindset

สำหรับ Data Engineer สิ่งที่ต้องจำคือ:

- `readStream` สร้าง streaming DataFrame พร้อม execution plan ส่วน query เริ่ม execute จริงเมื่อเรียก `writeStream.start()`
- checkpoint คือหัวใจของ restart/recovery
- Bronze streaming ควรเก็บ traceability ให้มากพอ
- trigger interval มีผลต่อ latency (trigger ถี่ = ข้อมูลถึง downstream เร็วกว่า), cost และ small files
- Silver/DQ/dedup logic ควรถูกออกแบบให้ handle duplicate ได้ และ rerun แล้วได้ผลเดิม
- production streaming ต้อง monitor ต่อเนื่อง ไม่ใช่รันแล้วปล่อยไว้เฉย ๆ

## Optional cleanup

In [ ]:
# for table_name in [bronze_table, silver_table, invalid_table, micro_batch_log_table]:
#     spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# if os.path.exists(base_path_mount):
#     shutil.rmtree(base_path_mount)

# print("Day 29 lab tables and files cleaned up.")